# Exercise 1: (50pts)

In this exercise, we try to visualize SGD. 

You are expected to repeate the same animation in this file using **plotly**: https://nbviewer.org/github/PhilChodrow/PIC16B/blob/master/lectures/math/optimization.ipynb

**Requirement:** 
1. Implement your own SGD algorithm with batch_size=1. 
2. You must use **plotly** to do the animation.


Grading is based on
1. The correctness of your code. (20pts)
2. Your explanation to your code. (20pts)
3. Your animation result. (10pts)

In [ ]:
# run this cell to generate data points
# please do NOT change it

m = 200

b0 = -0.2
b1 = 0.5

x = np.random.rand(m)
y = b0 + b1*x + 0.1*np.random.randn(m)

## Save and load trained model

Once you fully train your model, you expect to save it and load it for future use. You may also want to make your model public. Here, we discuss how to save and load trained neural network. 

One way to do that is calling `tf.keras.saving`. Here are the codes. Official documentation is [here](https://www.tensorflow.org/api_docs/python/tf/keras/saving/save_model).


In [ ]:
# save model
model.save("model.keras")

In [ ]:
# load model
new_model = tf.keras.models.load_model("model.keras")
new_model.evaluate(X_test, y_test)

Another way to do that is using checkpoint, the official documentation can be found through the link: https://www.tensorflow.org/tutorials/keras/save_and_load

In [ ]:
import os

checkpoint_path = "cp.ckpt"
checkpoint_dir = os.path.dirname(checkpoint_path)

# Create a callback that saves the model
cp_callback = tf.keras.callbacks.ModelCheckpoint(filepath=checkpoint_path,
                                                 save_weights_only=False,
                                                 verbose=1)

# construct the model
model = tf.keras.models.Sequential([
    layers.Dense(20, input_shape = (4,), activation='relu', kernel_initializer='he_normal'),
    layers.Dense(10, activation='relu',kernel_regularizer=tf.keras.regularizers.L1(0.01)),
    layers.Dense(3)                        
])
model.summary()

# loss function
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)

# set up optimizers:
optimizer_adam = tf.keras.optimizers.Adam(learning_rate=0.001, beta_1=0.9, beta_2=0.999, epsilon=1e-07, amsgrad=False)

# compile model
model.compile(optimizer = optimizer_adam,        # one sgd algorithm
              loss = loss_fn,            # loss function 
              metrics = ["accuracy"])    # 'accurancy' for classification and 'mse' for regression

# Train the model with the new callback
model.fit(X_train, 
          y_train,  
          epochs=10,
          callbacks=[cp_callback])  # Pass callback to training

# This may generate warnings related to saving the state of the optimizer.
# These warnings (and similar warnings throughout this notebook)
# are in place to discourage outdated usage, and can be ignored.

In [ ]:
# load model
new_model = tf.keras.models.load_model("model.keras")
new_model.evaluate(X_test, y_test)

## Early stopping

Early Stopping is another regularization technique that can be used to reduce overfitting in neural networks. But it is not limited to neural networks and can also be used with other machine learning models.

The idea behind early stopping is straightforward. We intentionally stop the training process of the model early right before the model begins to overfit.

The key challenge in early stopping is to identify the region where the model begins to overfit. For this, we need to monitor the training process. That can be achieved by plotting the training error and test (validation) error against the number of epochs during the training.

- Right Fit Region: The region where the test error stops decreasing (or begins increasing)

- Underfitting: before the right fit region, the model fails to learn the patterns in the training data and does not perform well. This is known as underfitting. 

- Overfitting: After the right fit region, the model tries to memorize the training data instead of learning patterns in the data. It does not generalize well on new unseen data (i.e. test data). That’s why the test error begins increasing while the training error still decreases.


Tensor flow documentation is [here](https://www.tensorflow.org/api_docs/python/tf/keras/callbacks/EarlyStopping).

In [ ]:
callback = tf.keras.callbacks.EarlyStopping(monitor='loss', patience=3)

# This callback will stop the training when there is no improvement in
# the loss for three consecutive epochs.

# monitor: Quantity to be monitored. Defaults to "val_loss".

model = tf.keras.models.Sequential([tf.keras.layers.Dense(1, input_shape=(20,))])
model.summary()
model.compile("SGD", loss='mse')
history = model.fit(np.arange(100).reshape(5, 20), np.zeros(5),
                    epochs=10, batch_size=1, callbacks=[callback],
                    verbose=1)
len(history.history['loss'])  # Only 4 epochs are run.
